# Canadian Household Spending Analysis — Comparative Core (2017, 2019, 2021)
**Analyst:** Carlos Restrepo | GLOCAL Foundation of Canada  
**Phase:** 04 — Comparative Core Analysis  
**Source:** Statistics Canada — SHS PUMF 2017, 2019, 2021  
**Kernel:** Python (shs2021)

## Purpose
This notebook creates a **single comparative layer** across 2017, 2019, and 2021 using the **real cleaned datasets and bootstrap outputs** generated in the yearly notebooks.

## Methodological notes
- This notebook is designed to be **more accurate than hardcoded comparisons**.  
  It reads directly from:
  - `shs_2017_clean.csv`
  - `shs_2019_clean.csv`
  - `shs_2021_clean.csv`
  - `bootstrap_ci_2017.csv`
  - `bootstrap_ci_2019.csv`
  - `bootstrap_ci_2021.csv`
- **2017 structural note:** the 2017 SHS PUMF uses separate **Interview** and **Diary** files.  
  The 2017 cleaning/analysis pipeline in this project uses the **Diary file only**, because it already contains the variables required for the expenditure analysis along with its own survey weights.
- **2017 Atlantic geography note:** in the 2017 PUMF, the four Atlantic provinces are grouped under code **`'14'`** and treated here as **Atlantic Provinces**.  
  Therefore, 2017 provincial results are **not fully comparable** to 2019/2021 at the individual Atlantic province level.
- To produce Canada-level estimates comparable to official SHS practice, territorial capitals are excluded from the provincial analysis where applicable.


In [ ]:
import os
# ============================================================
# Cell 2 - Imports and helper functions
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

def weighted_mean(df, var, weight='WeightD'):
    valid = df[[var, weight]].dropna()
    return (valid[var] * valid[weight]).sum() / valid[weight].sum()

def standardize_income_column(df):
    if 'HH_TotInc' in df.columns:
        df = df.rename(columns={'HH_TotInc': 'HH_TOTAL_INCOME'})
    elif 'HHTOTINC' in df.columns:
        df = df.rename(columns={'HHTOTINC': 'HH_TOTAL_INCOME'})
    elif 'HH_TOTINC' in df.columns:
        df = df.rename(columns={'HH_TOTINC': 'HH_TOTAL_INCOME'})
    else:
        raise KeyError('No household income column found.')
    return df

def ensure_column(df, candidates, target_name):
    for c in candidates:
        if c in df.columns:
            if c != target_name:
                df = df.rename(columns={c: target_name})
            return df
    raise KeyError(f'Could not find any of these columns: {candidates}')

def essential_burden_pct(df):
    return ((weighted_mean(df, 'SH001') + weighted_mean(df, 'FD001') + weighted_mean(df, 'TR001')) 
            / weighted_mean(df, 'HH_TOTAL_INCOME')) * 100

def shelter_burden_pct(df):
    return (weighted_mean(df, 'SH001') / weighted_mean(df, 'HH_TOTAL_INCOME')) * 100

def food_burden_pct(df):
    return (weighted_mean(df, 'FD001') / weighted_mean(df, 'HH_TOTAL_INCOME')) * 100

def transport_burden_pct(df):
    return (weighted_mean(df, 'TR001') / weighted_mean(df, 'HH_TOTAL_INCOME')) * 100

def residual_income(df):
    return weighted_mean(df, 'HH_TOTAL_INCOME') - (
        weighted_mean(df, 'SH001') + weighted_mean(df, 'FD001') + weighted_mean(df, 'TR001')
    )

print('OK Helpers ready.')

In [ ]:
# ============================================================
# Cell 3 - Load yearly clean datasets and bootstrap outputs
# ============================================================

base = Path(r"C:\Users\LENOVO\canadian_household_spending")

paths = {
    2017: {
        'clean': base / 'outputs' / '2017' / 'shs_2017_clean.csv',
        'bootstrap': base / 'outputs' / '2017' / 'bootstrap_ci_2017.csv',
    },
    2019: {
        'clean': base / 'outputs' / '2019' / 'shs_2019_clean.csv',
        'bootstrap': base / 'outputs' / '2019' / 'bootstrap_ci_2019.csv',
    },
    2021: {
        'clean': base / 'outputs' / '2021' / 'shs_2021_clean.csv',
        'bootstrap': base / 'outputs' / '2021' / 'bootstrap_ci_2021.csv',
    }
}

dfs = {}
bootstrap_dfs = {}

for year, p in paths.items():
    df = pd.read_csv(p['clean'])
    df = standardize_income_column(df)
    df = ensure_column(df, ['WeightD', 'WEIGHTD'], 'WeightD')
    df = ensure_column(df, ['PROV_NAME'], 'PROV_NAME')
    df = ensure_column(df, ['TENURE_NAME'], 'TENURE_NAME')
    df = ensure_column(df, ['INCOME_QUINTILE'], 'INCOME_QUINTILE')
    df['year'] = year
    df = df.copy()  # evita fragmentación excesiva
    dfs[year] = df

    if p['bootstrap'].exists():
        bdf = pd.read_csv(p['bootstrap'])
        bdf['year'] = year
        bootstrap_dfs[year] = bdf

    print(f"{year}: clean rows={df.shape[0]:,}, cols={df.shape[1]:,}")

print("\nBootstrap files loaded:")
for year, bdf in bootstrap_dfs.items():
    print(f"{year}: rows={bdf.shape[0]:,}, cols={bdf.shape[1]:,}")

In [ ]:
# ============================================================
# Cell 4 - Build harmonized comparative dataset
# ============================================================

common_cols = [
    'year', 'Prov', 'PROV_NAME', 'TENURE_NAME', 'INCOME_QUINTILE',
    'WeightD', 'HH_TOTAL_INCOME', 'FD001', 'SH001', 'TR001', 'HC001', 'TE001'
]

df_all = pd.concat([dfs[2017][common_cols], dfs[2019][common_cols], dfs[2021][common_cols]],
                   ignore_index=True)

print('OK Harmonized dataset created!')
print(df_all.shape)
print(df_all[['year', 'HH_TOTAL_INCOME', 'FD001', 'SH001', 'TR001', 'HC001', 'TE001']].head())

In [ ]:
# ============================================================
# Cell 5 - National comparative summary
# ============================================================

national_rows = []
for year in [2017, 2019, 2021]:
    d = dfs[year]
    national_rows.append({
        'Year': year,
        'Households': int(d['WeightD'].sum()),
        'Avg_Income': weighted_mean(d, 'HH_TOTAL_INCOME'),
        'Avg_TotalExp': weighted_mean(d, 'TE001'),
        'Avg_Food': weighted_mean(d, 'FD001'),
        'Avg_Shelter': weighted_mean(d, 'SH001'),
        'Avg_Transport': weighted_mean(d, 'TR001'),
        'Avg_Health': weighted_mean(d, 'HC001'),
        'Shelter_pct_income': shelter_burden_pct(d),
        'Food_pct_income': food_burden_pct(d),
        'Transport_pct_income': transport_burden_pct(d),
        'Essential_pct_income': essential_burden_pct(d),
        'Residual_Income': residual_income(d),
    })

national_df = pd.DataFrame(national_rows).round(1)

print("=== NATIONAL COMPARATIVE SUMMARY ===")
display(national_df)

In [ ]:
# ============================================================
# Cell 6 - Comparative analysis by income quintile
# ============================================================

q_rows = []
for year in [2017, 2019, 2021]:
    for q in ['Q1 - Lowest', 'Q2', 'Q3', 'Q4', 'Q5 - Highest']:
        subset = dfs[year][dfs[year]['INCOME_QUINTILE'] == q]
        q_rows.append({
            'Year': year,
            'Quintile': q,
            'Households': int(subset['WeightD'].sum()),
            'Avg_Income': weighted_mean(subset, 'HH_TOTAL_INCOME'),
            'Avg_Shelter': weighted_mean(subset, 'SH001'),
            'Avg_Food': weighted_mean(subset, 'FD001'),
            'Avg_Transport': weighted_mean(subset, 'TR001'),
            'Shelter_pct_income': shelter_burden_pct(subset),
            'Food_pct_income': food_burden_pct(subset),
            'Transport_pct_income': transport_burden_pct(subset),
            'Essential_pct_income': essential_burden_pct(subset),
            'Residual_Income': residual_income(subset),
        })

quintile_comp = pd.DataFrame(q_rows).round(1)

print("=== QUINTILE COMPARISON ===")
print(quintile_comp.to_string(index=False))

In [ ]:
# ============================================================
# Cell 7 - Comparative analysis by tenure
# ============================================================

t_rows = []
for year in [2017, 2019, 2021]:
    for tenure in ['Owner with mortgage', 'Owner without mortgage', 'Renter']:
        subset = dfs[year][dfs[year]['TENURE_NAME'] == tenure]
        t_rows.append({
            'Year': year,
            'Tenure': tenure,
            'Households': int(subset['WeightD'].sum()),
            'Avg_Income': weighted_mean(subset, 'HH_TOTAL_INCOME'),
            'Avg_Shelter': weighted_mean(subset, 'SH001'),
            'Avg_Food': weighted_mean(subset, 'FD001'),
            'Avg_Transport': weighted_mean(subset, 'TR001'),
            'Shelter_pct_income': shelter_burden_pct(subset),
            'Food_pct_income': food_burden_pct(subset),
            'Transport_pct_income': transport_burden_pct(subset),
            'Essential_pct_income': essential_burden_pct(subset),
            'Residual_Income': residual_income(subset),
        })

tenure_comp = pd.DataFrame(t_rows).round(1)

print("=== TENURE COMPARISON ===")
print(tenure_comp.to_string(index=False))

In [ ]:
# ============================================================
# Cell 8 - Provincial comparison
# ============================================================

prov_rows = []
for year in [2017, 2019, 2021]:
    d = dfs[year].copy()
    province_list = sorted(d['PROV_NAME'].dropna().unique())
    for prov in province_list:
        subset = d[d['PROV_NAME'] == prov]
        prov_rows.append({
            'Year': year,
            'Province': prov,
            'Households': int(subset['WeightD'].sum()),
            'Avg_Income': weighted_mean(subset, 'HH_TOTAL_INCOME'),
            'Avg_Shelter': weighted_mean(subset, 'SH001'),
            'Avg_Food': weighted_mean(subset, 'FD001'),
            'Avg_Transport': weighted_mean(subset, 'TR001'),
            'Shelter_pct_income': shelter_burden_pct(subset),
            'Food_pct_income': food_burden_pct(subset),
            'Transport_pct_income': transport_burden_pct(subset),
            'Essential_pct_income': essential_burden_pct(subset),
            'Residual_Income': residual_income(subset),
        })

prov_comp = pd.DataFrame(prov_rows).round(1)

print("=== PROVINCIAL COMPARISON ===")
print(prov_comp.to_string(index=False))

In [ ]:
# ============================================================
# Cell 9 - Bootstrap CI comparison (if available)
# ============================================================

if bootstrap_dfs:
    bootstrap_all = pd.concat(bootstrap_dfs.values(), ignore_index=True)
    print("=== BOOTSTRAP CI COMPARISON ===")
    print(bootstrap_all.to_string(index=False))
else:
    print("No bootstrap CSV files found.")

In [ ]:
# ============================================================
# Cell 10 - Chart: National averages by year
# ============================================================

plot_df = national_df.set_index('Year')[['Avg_Income', 'Avg_Shelter', 'Avg_Food', 'Avg_Transport']]

ax = plot_df.plot(kind='bar', figsize=(12, 6), edgecolor='white')
ax.set_title('National Averages by Year — SHS PUMF Comparative Core', fontweight='bold')
ax.set_ylabel('CAD ($)')
ax.set_xlabel('Year')
ax.tick_params(axis='x', rotation=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=8)

plt.tight_layout()
plt.savefig(base / 'outputs' / 'comparative_national_averages.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')

In [ ]:
# ============================================================
# Cell 11 - Chart: Essential spending burden by quintile and year
# ============================================================

q_plot = quintile_comp.pivot(index='Quintile', columns='Year', values='Essential_pct_income')

ax = q_plot.plot(kind='bar', figsize=(12, 6), edgecolor='white')
ax.set_title('Essential Spending Burden by Quintile and Year', fontweight='bold')
ax.set_ylabel('% of Household Income')
ax.set_xlabel('Income Quintile')
ax.tick_params(axis='x', rotation=0)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8)

plt.tight_layout()
plt.savefig(base / 'outputs' / 'comparative_essential_burden_quintile.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')

In [ ]:
# ============================================================
# Cell 12 - Chart: Renter vs owner shelter burden over time
# ============================================================

tenure_plot = tenure_comp.pivot(index='Tenure', columns='Year', values='Shelter_pct_income')

ax = tenure_plot.plot(kind='bar', figsize=(12, 6), edgecolor='white')
ax.set_title('Shelter Burden by Tenure and Year', fontweight='bold')
ax.set_ylabel('% of Household Income')
ax.set_xlabel('Tenure')
ax.tick_params(axis='x', rotation=15)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8)

plt.tight_layout()
plt.savefig(base / 'outputs' / 'comparative_tenure_shelter_burden.png', dpi=150, bbox_inches='tight')
plt.show()
print('OK Chart saved!')

In [ ]:
# ============================================================
# Cell 13 - Executive summary table
# ============================================================

summary_rows = []
for year in [2017, 2019, 2021]:
    d = dfs[year]
    q1 = quintile_comp[(quintile_comp['Year'] == year) & (quintile_comp['Quintile'] == 'Q1 - Lowest')].iloc[0]
    q5 = quintile_comp[(quintile_comp['Year'] == year) & (quintile_comp['Quintile'] == 'Q5 - Highest')].iloc[0]
    renter = tenure_comp[(tenure_comp['Year'] == year) & (tenure_comp['Tenure'] == 'Renter')].iloc[0]
    owner_m = tenure_comp[(tenure_comp['Year'] == year) & (tenure_comp['Tenure'] == 'Owner with mortgage')].iloc[0]
    top_shelter = prov_comp[prov_comp['Year'] == year].sort_values('Shelter_pct_income', ascending=False).iloc[0]

    summary_rows.append({
        'Year': year,
        'Avg_Income': weighted_mean(d, 'HH_TOTAL_INCOME'),
        'Avg_Shelter': weighted_mean(d, 'SH001'),
        'Top_Province_Shelter_Burden': top_shelter['Province'],
        'Q1_Shelter_Burden': q1['Shelter_pct_income'],
        'Q5_Shelter_Burden': q5['Shelter_pct_income'],
        'Q1_Essential_Burden': q1['Essential_pct_income'],
        'Q5_Essential_Burden': q5['Essential_pct_income'],
        'Renter_Shelter_Burden': renter['Shelter_pct_income'],
        'OwnerMortgage_Shelter_Burden': owner_m['Shelter_pct_income'],
        'Renter_Avg_Income': renter['Avg_Income'],
        'OwnerMortgage_Avg_Income': owner_m['Avg_Income'],
    })

executive_df = pd.DataFrame(summary_rows).round(1)

print("=== EXECUTIVE SUMMARY TABLE ===")
display(executive_df)

executive_df.to_csv(base / 'outputs' / 'comparative_executive_summary.csv', index=False)
print("\nOK Summary saved!")

## Interpretation Notes
- This notebook is intended to serve as the **comparative core** for later sector-specific notebooks.
- It avoids hardcoding final values and instead reads directly from the **real cleaned yearly datasets** and bootstrap outputs.
- **2017 results should be interpreted with extra care** because:
  - they are based on the **Diary file**,
  - the four Atlantic provinces are grouped under code **`'14'`**,
  - and some categories in the 2017 SHS design are treated as **mixed categories** in the official methodology.

In [ ]:
from pathlib import Path
import pandas as pd

path_2017 = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\2017\shs_2017_clean.csv")
path_2019 = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\2019\shs_2019_clean.csv")
path_2021 = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\2021\shs_2021_clean.csv")

df_2017 = pd.read_csv(path_2017)
df_2019 = pd.read_csv(path_2019)
df_2021 = pd.read_csv(path_2021)

print("2017:", df_2017.shape)
print("2019:", df_2019.shape)
print("2021:", df_2021.shape)

In [ ]:
df_2017["year"] = 2017
df_2019["year"] = 2019
df_2021["year"] = 2021

comparative_raw = pd.concat(
    [df_2017, df_2019, df_2021],
    ignore_index=True
)

print("Tabla comparativa unida:", comparative_raw.shape)
comparative_raw[["year"]].value_counts().sort_index()

In [ ]:
print("Columnas disponibles:")
for col in comparative_raw.columns:
    if any(x in col.upper() for x in ["QUINT", "INC", "WEIGHT", "SH", "FD", "TR", "PROV"]):
        print(col)

In [ ]:
# Crear una columna estándar de ingreso para los 3 años
# Usa HH_TotInc cuando exista, y si está vacío usa HHTOTINC

comparative_raw["income_standard"] = comparative_raw["HH_TotInc"]

if "HHTOTINC" in comparative_raw.columns:
    comparative_raw["income_standard"] = comparative_raw["income_standard"].fillna(comparative_raw["HHTOTINC"])

# Revisar si ya quedó ingreso para 2021
comparative_raw.groupby("year")["income_standard"].apply(lambda x: x.isna().sum())

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# 1. Rutas de los CSV limpios
# =========================

path_2017 = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\2017\shs_2017_clean.csv")
path_2019 = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\2019\shs_2019_clean.csv")
path_2021 = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\2021\shs_2021_clean.csv")

# =========================
# 2. Leer los CSV
# =========================

df_2017 = pd.read_csv(path_2017)
df_2019 = pd.read_csv(path_2019)
df_2021 = pd.read_csv(path_2021)

# =========================
# 3. Agregar columna year
# =========================

df_2017 = df_2017.copy()
df_2019 = df_2019.copy()
df_2021 = df_2021.copy()

df_2017["year"] = 2017
df_2019["year"] = 2019
df_2021["year"] = 2021

# =========================
# 4. Unir los tres años
# =========================

comparative_raw = pd.concat(
    [df_2017, df_2019, df_2021],
    ignore_index=True
)

print("Tabla comparativa unida:", comparative_raw.shape)
print(comparative_raw["year"].value_counts().sort_index())

# =========================
# 5. Crear ingreso estándar
# =========================
# 2017 y 2019 usan HH_TotInc.
# 2021 puede venir con HHTOTINC.
# Esta columna unifica el ingreso para los tres años.

comparative_raw["income_standard"] = comparative_raw["HH_TotInc"]

if "HHTOTINC" in comparative_raw.columns:
    comparative_raw["income_standard"] = comparative_raw["income_standard"].fillna(
        comparative_raw["HHTOTINC"]
    )

print("\nValores faltantes en income_standard por año:")
print(comparative_raw.groupby("year")["income_standard"].apply(lambda x: x.isna().sum()))

# =========================
# 6. Función de promedio ponderado
# =========================

def weighted_avg(g, value_col, weight_col="WeightD"):
    data = g[[value_col, weight_col]].dropna()
    if data.empty or data[weight_col].sum() == 0:
        return np.nan
    return np.average(data[value_col], weights=data[weight_col])

# =========================
# 7. Crear resumen comparativo por año y quintil
# =========================

comparative_summary = (
    comparative_raw
    .groupby(["year", "INCOME_QUINTILE"], as_index=False)
    .apply(lambda g: pd.Series({
        "avg_income": weighted_avg(g, "income_standard"),
        "avg_shelter": weighted_avg(g, "SH001"),
        "avg_food": weighted_avg(g, "FD001"),
        "avg_transport": weighted_avg(g, "TR001"),
        "avg_essential": (
            weighted_avg(g, "SH001") +
            weighted_avg(g, "FD001") +
            weighted_avg(g, "TR001")
        ),
        "shelter_burden": weighted_avg(g, "SH001") / weighted_avg(g, "income_standard"),
        "food_burden": weighted_avg(g, "FD001") / weighted_avg(g, "income_standard"),
        "transport_burden": weighted_avg(g, "TR001") / weighted_avg(g, "income_standard"),
        "essential_burden": (
            weighted_avg(g, "SH001") +
            weighted_avg(g, "FD001") +
            weighted_avg(g, "TR001")
        ) / weighted_avg(g, "income_standard")
    }))
    .reset_index(drop=True)
)

# =========================
# 8. Agregar residual income
# =========================
# Residual income = ingreso promedio - gasto esencial promedio

comparative_summary["residual_income"] = (
    comparative_summary["avg_income"] - comparative_summary["avg_essential"]
)

# =========================
# 9. Ordenar columnas
# =========================

comparative_summary = comparative_summary[
    [
        "year",
        "INCOME_QUINTILE",
        "avg_income",
        "avg_shelter",
        "avg_food",
        "avg_transport",
        "avg_essential",
        "shelter_burden",
        "food_burden",
        "transport_burden",
        "essential_burden",
        "residual_income"
    ]
]

# =========================
# 10. Exportar CSV comparativo
# =========================

output_dir = Path(r"C:\Users\LENOVO\canadian_household_spending\outputs\comparative")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "shs_comparative_summary_2017_2019_2021.csv"

comparative_summary.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\nCSV comparativo creado en:")
print(output_path)

# Mostrar tabla final
comparative_summary

In [ ]:
# =========================
# CSV nacional comparativo 2017-2019-2021
# Una fila por año
# =========================

national_summary = (
    comparative_raw
    .groupby(["year"], as_index=False)
    .apply(lambda g: pd.Series({
        "avg_income": weighted_avg(g, "income_standard"),
        "avg_shelter": weighted_avg(g, "SH001"),
        "avg_food": weighted_avg(g, "FD001"),
        "avg_transport": weighted_avg(g, "TR001"),
        "avg_essential": (
            weighted_avg(g, "SH001") +
            weighted_avg(g, "FD001") +
            weighted_avg(g, "TR001")
        ),
        "shelter_burden": weighted_avg(g, "SH001") / weighted_avg(g, "income_standard"),
        "food_burden": weighted_avg(g, "FD001") / weighted_avg(g, "income_standard"),
        "transport_burden": weighted_avg(g, "TR001") / weighted_avg(g, "income_standard"),
        "essential_burden": (
            weighted_avg(g, "SH001") +
            weighted_avg(g, "FD001") +
            weighted_avg(g, "TR001")
        ) / weighted_avg(g, "income_standard")
    }))
    .reset_index(drop=True)
)

national_summary["residual_income"] = (
    national_summary["avg_income"] - national_summary["avg_essential"]
)

output_path_national = output_dir / "shs_national_summary_2017_2019_2021.csv"

national_summary.to_csv(output_path_national, index=False, encoding="utf-8-sig")

print("CSV nacional creado en:")
print(output_path_national)

national_summary

In [ ]:
# =========================
# CSV provincial comparativo 2017-2019-2021
# Una fila por año y provincia
# =========================

province_summary = (
    comparative_raw
    .groupby(["year", "PROV_NAME"], as_index=False)
    .apply(lambda g: pd.Series({
        "avg_income": weighted_avg(g, "income_standard"),
        "avg_shelter": weighted_avg(g, "SH001"),
        "avg_food": weighted_avg(g, "FD001"),
        "avg_transport": weighted_avg(g, "TR001"),
        "avg_essential": (
            weighted_avg(g, "SH001") +
            weighted_avg(g, "FD001") +
            weighted_avg(g, "TR001")
        ),
        "shelter_burden": weighted_avg(g, "SH001") / weighted_avg(g, "income_standard"),
        "food_burden": weighted_avg(g, "FD001") / weighted_avg(g, "income_standard"),
        "transport_burden": weighted_avg(g, "TR001") / weighted_avg(g, "income_standard"),
        "essential_burden": (
            weighted_avg(g, "SH001") +
            weighted_avg(g, "FD001") +
            weighted_avg(g, "TR001")
        ) / weighted_avg(g, "income_standard")
    }))
    .reset_index(drop=True)
)

province_summary["residual_income"] = (
    province_summary["avg_income"] - province_summary["avg_essential"]
)

output_path_province = output_dir / "shs_province_summary_2017_2019_2021.csv"

province_summary.to_csv(output_path_province, index=False, encoding="utf-8-sig")

print("CSV provincial creado en:")
print(output_path_province)

province_summary

In [ ]:
import urllib
from sqlalchemy import create_engine, text

server   = os.environ.get("AZURE_SQL_SERVER", "your-server.database.windows.net")
database = os.environ.get("AZURE_SQL_DATABASE", "your_database")
username = os.environ.get("AZURE_SQL_USERNAME", "your_username")
password = os.environ.get("AZURE_SQL_PASSWORD", "YOUR_PASSWORD_HERE")

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER=tcp:{server},1433;"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    print(conn.execute(text("SELECT 1")).fetchall())

df_all.to_sql("comparative_all_raw", engine, if_exists="replace", index=False)
print(f"✓ comparative_all_raw: {df_all.shape}")

national_df.to_sql("comparative_national", engine, if_exists="replace", index=False)
print(f"✓ comparative_national: {national_df.shape}")

quintile_comp.to_sql("comparative_quintile", engine, if_exists="replace", index=False)
print(f"✓ comparative_quintile: {quintile_comp.shape}")

tenure_comp.to_sql("comparative_tenure", engine, if_exists="replace", index=False)
print(f"✓ comparative_tenure: {tenure_comp.shape}")

national_summary.to_sql("comparative_national_summary", engine, if_exists="replace", index=False)
print(f"✓ comparative_national_summary: {national_summary.shape}")

province_summary.to_sql("comparative_province_summary", engine, if_exists="replace", index=False)
print(f"✓ comparative_province_summary: {province_summary.shape}")